# 09 - Ternary Deployment Audit

Audita a diferença entre conversão funcional ternária e deploy ternário eficiente. Mede tamanho de artefatos, estima armazenamento ternário empacotado para MLPs e gera um plano de runtime.

In [ ]:
from pathlib import Path
import json
import math
import os

PROJECT_ROOT = Path.cwd() / 'gemma4_theory' if (Path.cwd() / 'gemma4_theory').exists() else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
OUT_DIR = RESULTS_DIR / 'deployment_audit'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)


In [ ]:
def dir_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return 0
    total = 0
    for p in path.rglob('*'):
        if p.is_file():
            total += p.stat().st_size
    return total

def human_bytes(n):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024

artifact_dirs = {
    'student_pruned': CHECKPOINTS_DIR / 'student_pruned',
    'student_ternary_recovered_l78_adapter': CHECKPOINTS_DIR / 'student_ternary_recovered_l78',
}

artifact_sizes = []
for name, path in artifact_dirs.items():
    size = dir_size_bytes(path)
    artifact_sizes.append({'artifact': name, 'path': str(path), 'size_bytes': size, 'size_human': human_bytes(size), 'exists': path.exists()})

artifact_sizes


In [ ]:
# Static estimate for Gemma-2-2B MLP ternary storage.
# Based on observed layer/module naming in the experiments: 26 layers, 3 MLP projections per layer.
# Gemma-2-2B hidden_size/intermediate_size are set here as configurable constants.

NUM_LAYERS = 26
HIDDEN_SIZE = 2304
INTERMEDIATE_SIZE = 9216
MLP_PROJECTIONS_PER_LAYER = 3

# gate: hidden -> intermediate
# up: hidden -> intermediate
# down: intermediate -> hidden
weights_per_layer = (HIDDEN_SIZE * INTERMEDIATE_SIZE) * 3
total_mlp_weights = NUM_LAYERS * weights_per_layer

fp16_bytes = total_mlp_weights * 2
int8_bytes = total_mlp_weights * 1
ternary_2bit_bytes = math.ceil(total_mlp_weights * 2 / 8)
ternary_with_scale_bytes = ternary_2bit_bytes + NUM_LAYERS * MLP_PROJECTIONS_PER_LAYER * 4

estimate = {
    'num_layers': NUM_LAYERS,
    'hidden_size': HIDDEN_SIZE,
    'intermediate_size': INTERMEDIATE_SIZE,
    'mlp_projections_per_layer': MLP_PROJECTIONS_PER_LAYER,
    'total_mlp_weights': total_mlp_weights,
    'fp16_mlp_bytes': fp16_bytes,
    'fp16_mlp_human': human_bytes(fp16_bytes),
    'int8_mlp_bytes': int8_bytes,
    'int8_mlp_human': human_bytes(int8_bytes),
    'ternary_2bit_mlp_bytes': ternary_2bit_bytes,
    'ternary_2bit_mlp_human': human_bytes(ternary_2bit_bytes),
    'ternary_2bit_with_per_projection_scale_bytes': ternary_with_scale_bytes,
    'ternary_2bit_with_per_projection_scale_human': human_bytes(ternary_with_scale_bytes),
    'theoretical_compression_vs_fp16': fp16_bytes / ternary_with_scale_bytes,
    'theoretical_compression_vs_int8': int8_bytes / ternary_with_scale_bytes,
}

estimate


In [ ]:
latest_l78 = sorted(RESULTS_DIR.glob('**/07_ternary_expansion_summary_*133727.json'))
if latest_l78:
    l78_summary = json.loads(latest_l78[-1].read_text(encoding='utf-8'))
else:
    l78_files = sorted(RESULTS_DIR.glob('**/07_ternary_expansion_summary_*.json'))
    l78_summary = json.loads(l78_files[-1].read_text(encoding='utf-8')) if l78_files else {}

audit = {
    'claim_supported_now': 'Functional conversion of all MLP projections to ternary-valued weights with recovery to local benchmark parity.',
    'claim_not_yet_supported': 'Efficient packed ternary runtime/deployment speedup.',
    'l78_summary': l78_summary,
    'artifact_sizes': artifact_sizes,
    'static_mlp_storage_estimate': estimate,
    'next_runtime_steps': [
        'Materialize merged l78 model or adapter-loadable reproducible checkpoint.',
        'Write exporter that stores MLP ternary signs/masks packed into 2-bit representation plus per-projection scales.',
        'Write reference Python unpack + matmul correctness test against current fp tensor outputs.',
        'Benchmark packed size on disk versus fp16 checkpoint and adapter representation.',
        'Only then implement or target real kernels: custom PyTorch extension, BitBLAS/T-MAC-style backend, or bitnet.cpp-compatible path if architecture constraints allow.',
    ],
}

out_path = OUT_DIR / 'ternary_deployment_audit.json'
out_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
print('Audit salvo em:', out_path)
print(json.dumps(audit, ensure_ascii=False, indent=2)[:6000])


## Interpretação

Este notebook separa o resultado já provado do resultado de deploy. Se o audit mostra paridade funcional em `l78`, o próximo trabalho é empacotamento e runtime, não mais recovery MLP.